# ECE326 Tutorial 4 - Midterm Solutions

Work through the October 22 midterm, then run the examples to check your reasoning.
This notebook follows the final exam's question numbering. Run it from top to
bottom; Python 3 and NumPy are all that is required. The examples illustrate the
solutions, not additional exam requirements.


In [ ]:
import numpy as np


## Q1 - True or false

| Item | Statement | Answer and reason |
|------|-----------|-------------------|
| 1 | If a and b refer to the same list, assigning a new list to b also makes a refer to the new list. | False. Rebinding b does not change a. |
| 2 | A Python list can be used as a dictionary key. | False. Lists are mutable and unhashable. |
| 3 | A function can append to a list passed as an argument, and the caller can see the change even if the function returns None. | True. The function can mutate the shared object. |
| 4 | For an array A of shape (3,4), np.add.reduce(A, axis=1) returns one total per column. | False. It returns three row totals. |
| 5 | Associativity alone permits arbitrary reordering of operands in a reduction. | False. Associativity permits regrouping, not arbitrary permutation. |
| 6 | For a writable NumPy array a and a same-shaped Boolean mask, assigning a scalar to a[mask] changes only entries where mask is True. | True. Boolean assignment selects the entries to change. |
| 7 | A shallow copy of a list containing lists still shares the inner lists with the original. | True. Only the outer list is copied. |
| 8 | If class C inherits from class B, then C itself is necessarily an instance of B. | False. Inheritance and instance-of are different relations. |
| 9 | When an inherited method calls self.f(), an override of f in the subclass can be invoked. | True. Method lookup uses the actual instance. |
| 10 | Python applies a function's decorator when it executes the function definition, even if that function is never called. | True. Decoration and calling the resulting function are separate events. |


### References, mutation and copying
Rebinding a name is different from modifying the object to which it refers.


In [ ]:
a = [1]
b = a
b = [2]
assert a == [1] and b == [2]

def append_value(values):
    values.append(3)

result = append_value(a)
assert result is None and a == [1, 3]

outer = [[1]]
copied = outer.copy()
copied[0].append(2)
assert outer is not copied and outer[0] is copied[0]
print("Shared inner list:", outer, copied)


### Reduction and masking
Axis 1 runs across the columns within each row. String concatenation provides
a counterexample to arbitrary reordering: it is associative but not commutative.


In [ ]:
A = np.arange(12).reshape(3, 4)
print(A)
print("Row totals:", np.add.reduce(A, axis=1))
assert np.array_equal(np.add.reduce(A, axis=1), [6, 22, 38])
assert ("a" + "b") + "c" == "a" + ("b" + "c")
assert "a" + "b" + "c" != "b" + "a" + "c"

a = np.array([1, 2, 3, 4])
a[a > 2] = 0
assert np.array_equal(a, [1, 2, 0, 0])
print("Masked assignment:", a)


### Inheritance and decoration
An instance of C is also an instance of B, but the class object C is not itself
an instance of B in this ordinary class hierarchy.


In [ ]:
class B:
    def f(self):
        return "B"
    def call_f(self):
        return self.f()

class C(B):
    def f(self):
        return "C"

assert issubclass(C, B) and not isinstance(C, B)
assert isinstance(C(), B)
assert C().call_f() == "C"

events = []
def register(f):
    events.append("decorated")
    return f

@register
def greeting():
    events.append("called")

assert events == ["decorated"]
greeting()
assert events == ["decorated", "called"]
print(events)


## Q2 - Score histories
Implement Scores: adding two histories concatenates their lists. An integer on
the left is prepended; one on the right is appended. Construction and addition
must create independent lists without changing the operands. Empty lists are
allowed. Return NotImplemented for unsupported operands; booleans and subclasses
are outside the question's scope.

The constructor copies its input. Addition constructs a new Scores object.
Reflected addition must preserve operand order: 2 ~~ a is not a ~~ 2.


In [ ]:
class Scores:
    def __init__(self, values):
        self.values = list(values)
    def __add__(self, other):
        if isinstance(other, Scores):
            return Scores(self.values + other.values)
        if isinstance(other, int):
            return Scores(self.values + [other])
        return NotImplemented
    def __radd__(self, other):
        if isinstance(other, Scores):
            return Scores(other.values + self.values)
        if isinstance(other, int):
            return Scores([other] + self.values)
        return NotImplemented


The Scores branch in __radd__ is not needed for ordinary Scores + Scores;
Python uses __add__ in that case. NotImplemented is a return value, not an
exception to raise.


In [ ]:
a = Scores([4, 7])
b = Scores([9])
assert (a + b).values == [4, 7, 9]
assert (a + 2).values == [4, 7, 2]
assert (2 + a).values == [2, 4, 7]
assert a.values == [4, 7] and b.values == [9]
assert a.__add__("x") is NotImplemented
assert a.__radd__("x") is NotImplemented
assert (Scores([]) + a).values == [4, 7]
print("Left integer:", (2 + a).values)
print("Right integer:", (a + 2).values)


### Q2(b) - Trace the lists
Changing raw cannot change a.values because construction copied the list.
Changing result.values cannot change a.values because addition created a new
history. The answers are [4, 7] and [2, -1, 7, 9].


In [ ]:
raw = [4, 7]
a = Scores(raw)
result = 2 + a + Scores([9])
raw[0] = 99
result.values[1] = -1
assert a.values == [4, 7]
assert result.values == [2, -1, 7, 9]
print("a.values:", a.values)
print("result.values:", result.values)


## Q3 - A call-limit decorator
Write limit_calls(k), for integer k >= 0. Each decorated function may execute
its first k calls, forwarding all positional and keyword arguments and returning
the result unchanged. Later calls return None without invoking the function.
Every permitted call counts, including one returning None. Each function gets
its own allowance, even when a decorator is reused. Functions are nonrecursive
and finish normally.

There are three levels: limit_calls captures k, decorate receives the function,
and wrapper handles each call. Store the count on the wrapper so each decorated
function gets separate state. Do not reset the count inside each invocation.


In [ ]:
def limit_calls(k):
    def decorate(f):
        def wrapper(*args, **kwargs):
            if wrapper.calls >= k:
                return None
            wrapper.calls += 1
            return f(*args, **kwargs)
        wrapper.calls = 0
        return wrapper
    return decorate


In [ ]:
entries = []

@limit_calls(2)
def record(value):
    entries.append(value)
    return value

assert record(7) == 7
assert record(value=None) is None
assert record(9) is None
assert entries == [7, None]
print("Actually recorded:", entries)

decorate = limit_calls(1)
first = decorate(lambda: "first")
second = decorate(lambda: "second")
assert first() == "first" and first() is None
assert second() == "second" and second() is None
never = limit_calls(0)(lambda: entries.append(99))
assert never() is None and entries == [7, None]
print("Independent allowances and zero limit: passed")


## Q4 - Keep moving in a directed graph
A Boolean array A of shape (n,n), n >= 1, represents directed edges: A[i,j]
means a move from i to j is possible. Return one Boolean per starting location,
without changing A. Revisiting locations and edges is allowed. No matrix
multiplication or graph libraries; parts (a) and (b) use no loops/comprehensions,
and part (c) may loop over updates, not individual vertices or edges.

Use this graph throughout: 0 -> 1, 1 -> 2, 2 -> 1, and 3 -> 4. Locations
1 and 2 form a cycle; 0 can reach it. Location 4 is a dead end.


In [ ]:
A = np.array([
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0],
], dtype=bool)
original_A = A.copy()


### Q4(a) - At least one move
OR across the destination columns. A row is True if any outgoing edge exists.


In [ ]:
def one_move(A):
    return np.logical_or.reduce(A, axis=1)

one = one_move(A)
assert np.array_equal(one, [True, True, True, True, False])
print("One move:", one)


### Q4(b) - Two successive moves
The first move must reach a location that itself has an outgoing edge.
The mask one selects those destination columns, not starting rows.
one[None, :] has shape (1,n), so each row uses the same destination mask.
Writing A & one gives the same result through broadcasting.


In [ ]:
def two_moves(A):
    one = one_move(A)
    return np.logical_or.reduce(A & one[None, :], axis=1)

two = two_moves(A)
assert np.array_equal(two, [True, True, True, False, False])
print("Two moves:", two)


### Q4(c) - An indefinitely continuing walk
Keep a location if it can move into the current set of candidates. Start with
every location, then repeat until the set stops changing. This asks whether
some choice of moves can continue forever, not whether every choice can.


In [ ]:
def viable_locations(A):
    viable = np.ones(A.shape[0], dtype=bool)
    while True:
        updated = np.logical_or.reduce(A & viable[None, :], axis=1)
        if np.array_equal(updated, viable):
            return updated
        viable = updated

assert np.array_equal(viable_locations(A), [True, True, True, False, False])
assert np.array_equal(A, original_A)
print("Can keep moving:", viable_locations(A))


Why it stops: after t updates, the mask identifies locations from which a walk
of at least t moves exists. These sets can only shrink. Each changed update
removes at least one location, so at most n strict changes occur.

Why the answer is correct: in the stable set, every surviving location has a
successor in that same set. We can therefore keep moving. Conversely, a location
on an infinite walk is never removed: its next location also permits infinite
continuation. Thus no valid starting location is lost.

A location need not itself lie on a cycle; reaching one is enough. An optional
edge to a dead end does not spoil another choice that permits infinite movement.


In [ ]:
branch = A.copy()
branch[0, 4] = True
assert np.array_equal(viable_locations(branch), viable_locations(A))
assert np.array_equal(viable_locations(np.array([[True]])), [True])
assert np.array_equal(viable_locations(np.array([[False]])), [False])
chain = np.array([[0, 1, 0], [0, 0, 1], [0, 0, 0]], dtype=bool)
assert not viable_locations(chain).any()
print("Cycle, dead end, self-loop and chain checks: passed")
print("All tutorial checks passed")
